In [1]:
from pprint import pprint

import torch
from huggingface_hub import HfApi

import lerobot
from lerobot.common.datasets.lerobot_dataset import LeRobotDataset, LeRobotDatasetMetadata

/opt/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

# We ported a number of existing datasets ourselves, use this to see the list:
print("List of available datasets:")


# Let's take this one for this example
repo_id = "cpw/test_disp"
# We can have a look and fetch its metadata to know more about it:
ds_meta = LeRobotDatasetMetadata(repo_id)

# By instantiating just this class, you can quickly access useful information about the content and the
# structure of the dataset without downloading the actual data yet (only metadata files — which are
# lightweight).
print(f"Total number of episodes: {ds_meta.total_episodes}")
print(f"Average number of frames per episode: {ds_meta.total_frames / ds_meta.total_episodes:.3f}")
print(f"Frames per second used during data collection: {ds_meta.fps}")
print(f"Robot type: {ds_meta.robot_type}")
print(f"keys to access images from cameras: {ds_meta.camera_keys=}\n")

print("Tasks:")
print(ds_meta.tasks)
print("Features:")
pprint(ds_meta.features)

# You can also get a short summary by simply printing the object:
print(ds_meta)

# You can then load the actual dataset from the hub.
# Either load any subset of episodes:
dataset = LeRobotDataset(repo_id, episodes=[0, 1, 2, 3])

# And see how many frames you have:
print(f"Selected episodes: {dataset.episodes}")
print(f"Number of episodes selected: {dataset.num_episodes}")
print(f"Number of frames selected: {dataset.num_frames}")

# Or simply load the entire dataset:
dataset = LeRobotDataset(repo_id)
print(f"Number of episodes selected: {dataset.num_episodes}")
print(f"Number of frames selected: {dataset.num_frames}")

# The previous metadata class is contained in the 'meta' attribute of the dataset:
print(dataset.meta)

# LeRobotDataset actually wraps an underlying Hugging Face dataset
# (see https://huggingface.co/docs/datasets for more information).
print(dataset.hf_dataset)

# LeRobot datasets also subclasses PyTorch datasets so you can do everything you know and love from working
# with the latter, like iterating through the dataset.
# The __getitem__ iterates over the frames of the dataset. Since our datasets are also structured by
# episodes, you can access the frame indices of any episode using the episode_data_index. Here, we access
# frame indices associated to the first episode:
episode_index = 0
from_idx = dataset.episode_data_index["from"][episode_index].item()
to_idx = dataset.episode_data_index["to"][episode_index].item()

# Then we grab all the image frames from the first camera:
camera_key = dataset.meta.camera_keys[0]
frames = [dataset[idx][camera_key] for idx in range(from_idx, to_idx)]

# The objects returned by the dataset are all torch.Tensors
print(type(frames[0]))
print(frames[0].shape)

# Since we're using pytorch, the shape is in pytorch, channel-first convention (c, h, w).
# We can compare this shape with the information available for that feature
pprint(dataset.features[camera_key])
# In particular:
print(dataset.features[camera_key]["shape"])
# The shape is in (h, w, c) which is a more universal format.

# For many machine learning applications we need to load the history of past observations or trajectories of
# future actions. Our datasets can load previous and future frames for each key/modality, using timestamps
# differences with the current loaded frame. For instance:
delta_timestamps = {
    # loads 4 images: 1 second before current frame, 500 ms before, 200 ms before, and current frame
    camera_key: [-1, -0.5, -0.20, 0],
    # loads 6 state vectors: 1.5 seconds before, 1 second before, ... 200 ms, 100 ms, and current frame
    "observation.state": [-1.5, -1, -0.5, -0.20, -0.10, 0],
    # loads 64 action vectors: current frame, 1 frame in the future, 2 frames, ... 63 frames in the future
    "action": [t / dataset.fps for t in range(64)],
}
# Note that in any case, these delta_timestamps values need to be multiples of (1/fps) so that added to any
# timestamp, you still get a valid timestamp.

dataset = LeRobotDataset(repo_id, delta_timestamps=delta_timestamps)
print(f"\n{dataset[0][camera_key].shape=}")  # (4, c, h, w)
print(f"{dataset[0]['observation.state'].shape=}")  # (6, c)
print(f"{dataset[0]['action'].shape=}\n")  # (64, c)

# Finally, our datasets are fully compatible with PyTorch dataloaders and samplers because they are just
# PyTorch datasets.
dataloader = torch.utils.data.DataLoader(
    dataset,
    num_workers=0,
    batch_size=32,
    shuffle=True,
)

for batch in dataloader:
    print(f"{batch[camera_key].shape=}")  # (32, 4, c, h, w)
    print(f"{batch['observation.state'].shape=}")  # (32, 6, c)
    print(f"{batch['action'].shape=}")  # (32, 64, c)
    break


List of available datasets:
Total number of episodes: 5
Average number of frames per episode: 445.600
Frames per second used during data collection: 30
Robot type: DVRK
keys to access images from cameras: ds_meta.camera_keys=['observation.images.cam_left', 'observation.images.cam_right']

Tasks:
{0: 'Retraction and cutting'}
Features:
{'action': {'dtype': 'float32',
            'names': [['psm_yaw_joint',
                       'psm_pitch_end_joint',
                       'psm_main_insertion_joint',
                       'psm_tool_roll_joint',
                       'psm_tool_pitch_joint',
                       'psm_tool_yaw_joint',
                       'psm_tool_gripper_joint']],
            'shape': (7,)},
 'episode_index': {'dtype': 'int64', 'names': None, 'shape': (1,)},
 'frame_index': {'dtype': 'int64', 'names': None, 'shape': (1,)},
 'index': {'dtype': 'int64', 'names': None, 'shape': (1,)},
 'observation.dissection_tar': {'dtype': 'float32',
                               

/workspace/lerobot/common/datasets/compute_stats.py:142: RuntimeWarning: invalid value encountered in subtract
  delta_means = means - total_mean


Selected episodes: [0, 1, 2, 3]
Number of episodes selected: 4
Number of frames selected: 1806


/workspace/lerobot/common/datasets/compute_stats.py:142: RuntimeWarning: invalid value encountered in subtract
  delta_means = means - total_mean


Number of episodes selected: 5
Number of frames selected: 2228
LeRobotDatasetMetadata({
    Repository ID: 'cpw/test_disp',
    Total episodes: '5',
    Total frames: '2228',
    Features: '['observation.state', 'action', 'observation.point_cloud', 'observation.dissection_tar', 'observation.images.cam_left', 'observation.images.cam_right', 'timestamp', 'frame_index', 'episode_index', 'index', 'task_index']',
})',

Dataset({
    features: ['observation.state', 'action', 'observation.point_cloud', 'observation.dissection_tar', 'timestamp', 'frame_index', 'episode_index', 'index', 'task_index'],
    num_rows: 2228
})
<class 'torch.Tensor'>
torch.Size([3, 480, 640])
{'dtype': 'video',
 'info': {'has_audio': False,
          'video.channels': 3,
          'video.codec': 'av1',
          'video.fps': 30,
          'video.height': 480,
          'video.is_depth_map': False,
          'video.pix_fmt': 'yuv420p',
          'video.width': 640},
 'names': ['channels', 'height', 'width'],
 'shape'

In [3]:
import pyvista as pv
import numpy as np

# ------------------------------------------------------------------
# POINT CLOUD DATA  (replace this with your own array)
pts = batch['observation.point_cloud']      # shape (T, 4096, 3)
# ------------------------------------------------------------------

def save_pointcloud_video(points_3d: np.ndarray,
                          out_path: str = "pointcloud_traj.mp4",
                          fps: int = 10,
                          point_size: int = 5):
    """
    points_3d : np.ndarray  (T, N, 3)
    out_path  : str         – output video filename
    fps       : int         – frames per second
    point_size: int         – size of each rendered point
    """
    assert points_3d.ndim == 3 and points_3d.shape[2] == 3, "Expected (T, N, 3)"

    # Works even in headless notebooks 
    plotter = pv.Plotter() 
    plotter.open_movie(out_path, framerate=fps) 

    # Fix the camera once using the first frame
    mesh = pv.PolyData(points_3d[0].numpy())
    plotter.add_points(mesh, point_size=point_size,
                       render_points_as_spheres=False)
    # plotter.show(auto_close=False)          # draw 1st frame & lock camera
    plotter.write_frame()                   # write 1st frame
    plotter.clear()

    for frame_pts in points_3d[1:]:
        mesh.points = frame_pts.numpy()
        plotter.render()
        plotter.iren.process_events()
        plotter.write_frame()

    plotter.close()
    print(f"Saved → {out_path}")

# ------------------------------------------------------------------
save_pointcloud_video(pts, out_path="trajectory.mp4", fps=5, point_size=40)


libGL error: No matching fbConfigs or visuals found
libGL error: failed to load driver: swrast
libGL error: No matching fbConfigs or visuals found
libGL error: failed to load driver: swrast
2025-07-30 19:41:10.336 (   1.088s) [    7FEE7DC51000]vtkXOpenGLRenderWindow.:679   WARN| vtkXOpenGLRenderWindow (0x55f955f072c0): Cannot create GLX context.
2025-07-30 19:41:10.336 (   1.088s) [    7FEE7DC51000]vtkOpenGLRenderWindow.c:793   WARN| vtkXOpenGLRenderWindow (0x55f955f072c0): Failed to initialize OpenGL functions!
2025-07-30 19:41:10.336 (   1.088s) [    7FEE7DC51000]vtkOpenGLRenderWindow.c:812   WARN| vtkXOpenGLRenderWindow (0x55f955f072c0): Unable to find a valid OpenGL 3.2 or later implementation. Please update your video card driver to the latest version. If you are using Mesa please make sure you have version 11.2 or later and make sure your driver in Mesa supports OpenGL 3.2 such as llvmpipe or openswr. If you are on windows and using Microsoft remote desktop note that it only supp

Saved → trajectory.mp4
